# ДЗ 3. NumPy на данных «Дала Маркет»
Срок: **до четверга 12.11, 19:00**.
Сдача — папка `hw03` репозитория `dsa-homework` (`hw03.ipynb`): `git add hw03` → `git commit` → `git push`.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем — только массивы и маски (кроме задачи 8).

- **[База]** обязательно для всех · **[Стандарт]** — сколько получится · **[Челлендж]** — для тех, кому легко.

## [Для всех] Чтение кода — без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
# 1
a = np.arange(6).reshape(2, 3); print(a[1, 2], a.shape)
# 2
a = np.array([3, 8, 1, 8]); print(a.argmax(), a.max())
# 3
a = np.arange(10); b = a[::3]; b[0] = 99; print(a[0])
# 4
a = np.array([1, 5, 7, 2]); print((a > 3).sum(), (a > 3).mean())
# 5
print(np.zeros((2, 3)).shape, np.linspace(0, 1, 5)[1])
```

Мои предсказания и объяснения:

1. 5 (2,3) 
Получаем [0,1,2,3,4,5] -> reshape дает [[0,1,2][3,4,5]] -> получаем 5 (первая строка, третий столбец) и (кол-во строк, кол-во столбцов)
2. 1 8
argmax() выводит индекс первого вхождения максимального значения. max() выводит максимальное значение
3. 99
a = [0,1,2,3,4,5,6,7,8,9] -> b=[0,3,6,9] -> срез = ссылка поэтому 99
4. 2 0.5
Сумма по маске и среднее по маске. 2 значения больше 3 (5,7) -> 2/4 = 0.5
5. (2,3) 0.25
2 строки и 3 столбца у [[0,0,0],[0,0,0]] и элемент с индексом 1 из [0, 0.25, 0.5, 0.75, 1]

In [93]:
a = np.arange(6).reshape(2, 3); print(a[1, 2], a.shape)
# 2
a = np.array([3, 8, 1, 8]); print(a.argmax(), a.max())
# 3
a = np.arange(10); b = a[::3]; b[0] = 99; print(a[0])
# 4
a = np.array([1, 5, 7, 2]); print((a > 3).sum(), (a > 3).mean())
# 5
print(np.zeros((2, 3)).shape, np.linspace(0, 1, 5)[1])

5 (2, 3)
1 8
99
2 0.5
(2, 3) 0.25


In [94]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [95]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

# функции из урока 2 + превращаем колонки в массивы NumPy
amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])

print(amount.shape, amount.dtype, paid.dtype)

(5463,) float64 bool


## [База]
1. Найдите позицию (`argmax`) самого дорогого заказа. Какая сумма, клиент и город?
2. Выручка оплаченных заказов и сумма неоплаченных — через маску `paid` и `~paid`.
3. Сколько заказов в Шымкенте и какая у них доля от всех заказов?

In [96]:
# База
#1. Найдите позицию (`argmax`) самого дорогого заказа. Какая сумма, клиент и город?
argmax_index = amount.argmax()
print(amount.max())
print(cust[argmax_index])
print(city[argmax_index])


9999990.0
TEST
Алматы


In [97]:
#2. Выручка оплаченных заказов и сумма неоплаченных — через маску `paid` и `~paid`.
print(amount[paid].sum(), amount[~paid].sum(), sep='\n')


101909360.0
7974000.0


In [98]:
#3. Сколько заказов в Шымкенте и какая у них доля от всех заказов?
shymkent_cnt = (city == 'Шымкент').sum()
print(shymkent_cnt/len(city)*100)

14.71718835804503


## [Стандарт]
4. Маркетинг предлагает скидку 15% на оплаченные заказы дороже 50 000 ₸. Сколько выручки магазин недополучит?
   Посчитайте дважды: со всеми заказами и без заказов клиента TEST. Почему ответы различаются в несколько раз?
5. Для **каждого** города — доля заказов, доставленных за 1–2 дня. Список городов: `np.unique(city)`; цикл по 6 городам здесь можно.
6. Ниже код, который AI написал на запрос «посчитай средний чек». Найдите две ошибки, исправьте и посчитайте правильно:
```python
avg = amount.mean()
print("Средний чек:", round(avg))
```

In [99]:
# Стандарт
#4. Маркетинг предлагает скидку 15% на оплаченные заказы дороже 50 000 ₸. Сколько выручки магазин недополучит?
# Посчитайте дважды: со всеми заказами и без заказов клиента TEST. Почему ответы различаются в несколько раз?
discount_amount_50k = amount[(amount > 50000) & (paid)] * 0.85
discount_amount_50k_no_test = amount[(amount > 50000) & (paid) & (cust != 'TEST')] * 0.85
amount_50k = amount[(amount > 50000) & (paid)]

print(amount_50k.sum() - discount_amount_50k.sum(), amount_50k.sum() - discount_amount_50k_no_test.sum())


5702422.5 31202397.0


Ответ: потому что тестовый клиент имеет аномально большие чеки, сумма которых дает большую ошибку при агрегации без фильтрации.

In [100]:
#5. Для **каждого** города — доля заказов, доставленных за 1–2 дня. Список городов: `np.unique(city)`; цикл по 6 городам здесь можно.
for c in np.unique(city):
    fast_delievery = len(days[(city == c) & (days >= 2)])
    all_delievery = len(days[(city == c)])
    print(c, fast_delievery / all_delievery * 100)



Актобе 69.26952141057934
Алматы 64.25959780621572
Астана 65.82733812949641
Атырау 67.12328767123287
Караганда 61.152416356877325
Шымкент 81.96517412935323


In [101]:
#6. Ниже код, который AI написал на запрос «посчитай средний чек». Найдите две ошибки, исправьте и посчитайте правильно:
avg = amount[(paid) & (cust != 'TEST')].mean()
print("Средний чек:", round(avg, 2))


Средний чек: 14672.39


## [Челлендж]
7. Сравните скорость: сумма оплаченных заказов циклом `for` и через маску. Во сколько раз быстрее?
8. Смоделируйте A/B-тест. `np.random.seed(42)`. Группа A: 1 000 клиентов, вероятность покупки 5%; группа B: 1 000 клиентов, 6%
   (покупка = `np.random.rand(1000) < 0.05`). Сколько покупок в каждой группе?
   Повторите эксперимент 100 раз: в скольких случаях B оказалась лучше A? Что это говорит о размере выборки? (2–3 предложения)

In [102]:
# Челлендж
#7. Сравните скорость: сумма оплаченных заказов циклом `for` и через маску. Во сколько раз быстрее?
import time

t = time.time()
print('numpy:', amount[paid].sum())
t_numpy = time.time() - t

t = time.time()
summa = 0
for pay in amount:
    summa += pay
print('for loop:', summa)
t_loop = time.time() - t

print(round(t_loop/t_numpy, 2))

numpy: 101909360.0
for loop: 109883360.0
3.68


In [103]:
#8.Смоделируйте A/B-тест. `np.random.seed(42)`. Группа A: 1 000 клиентов, вероятность покупки 5%; группа B: 1 000 клиентов, 6%
# (покупка = `np.random.rand(1000) < 0.05`). Сколько покупок в каждой группе?
# Повторите эксперимент 100 раз: в скольких случаях B оказалась лучше A? Что это говорит о размере выборки? (2–3 предложения)
np.random.seed(42)
a_cnt, b_cnt, result = 0, 0, 0
for i in range(100):
    a = (np.random.rand(1000) < 0.05).sum()
    b = (np.random.rand(1000) < 0.06).sum()
    a_cnt += a
    b_cnt += b
    if b > a:
        result += 1
print(f"кол-во заказов у группы А = {a_cnt};\nкол-во заказов у группы B = {b_cnt};\nкол-во экспериментов которых B больше = {result};")



кол-во заказов у группы А = 4960;
кол-во заказов у группы B = 5938;
кол-во экспериментов которых B больше = 84;


Ответ: слишком маленькие группы для такого эксперимента. Чем больше будут группы тем выигрышнее будет 6%.